
# CH5 - Exercise 8

**(a)** Generate the data:
​```python
rng = np.random.default_rng(1)
x = rng.normal(size=100)
y = x - 2 * x**2 + rng.normal(size=100)
​```
Identify $n$ and $p$, and write the data-generating model in equation form.

**(b)** Scatterplot $X$ vs $Y$. Comment on the shape.

**(c)** Set a random seed and compute **LOOCV** errors for four least-squares fits:

1. $Y = \beta_0 + \beta_1 X + \epsilon$
2. $Y = \beta_0 + \beta_1 X + \beta_2 X^2 + \epsilon$
3. $Y = \beta_0 + \beta_1 X + \beta_2 X^2 + \beta_3 X^3 + \epsilon$
4. $Y = \beta_0 + \beta_1 X + \beta_2 X^2 + \beta_3 X^3 + \beta_4 X^4 + \epsilon$

**(d)** Repeat (c) with a different seed. Same results? Why?

**(e)** Which model has the smallest LOOCV error? Expected?

**(f)** Comment on the statistical significance of the coefficients from least squares. Do those conclusions agree with the CV results?

In [1]:
import numpy as np
import pandas as pd
from matplotlib .pyplot import subplots
import statsmodels .api as sm
from ISLP import load_data
from ISLP.models import ( ModelSpec as MS ,
summarize, poly)

from ISLP import confusion_table
from ISLP.models import contrast
from sklearn. discriminant_analysis import \
( LinearDiscriminantAnalysis as LDA ,
QuadraticDiscriminantAnalysis as QDA)
from sklearn. naive_bayes import GaussianNB
from sklearn. neighbors import KNeighborsClassifier
from sklearn. preprocessing import StandardScaler
from sklearn. model_selection import train_test_split
from sklearn. linear_model import LogisticRegression

from functools import partial
from sklearn.model_selection import (
    train_test_split, cross_validate, KFold, ShuffleSplit
)
from sklearn import clone 
from ISLP.models import sklearn_sm


### Model:

$$Y = X - 2\cdot X^{2} + \epsilon$$ 
with $\epsilon \sim \mathcal{N}(0,1)$

In [ ]:
rng = np.random.default_rng(1)
x = rng.normal(size=100)
y = x -2*x**2 + rng.normal(size=100) 

In [ ]:
fig,ax = subplots(figsize=(4,4))
ax.scatter(x,y,c='g',marker='x')
ax.set_xlabel("X")
ax.set_ylabel("Y")

We create a data frame containing x,y

In [ ]:
data = pd.DataFrame({'x':x,'y':y})
data.head()

In [29]:
n = 1000
# generate the data 
rng = np.random.default_rng(10)
x = rng.normal(size=n)
y = x -2*x**2 + rng.normal(size=n) 

# creation fo the dataset 
data = pd.DataFrame({'x':x,'y':y})
data.head()
#

# loop over the possible models
degree = 4
for deg in range(1,degree+1):
    design = MS([poly('x',degree=deg)])
    X_tot = design.fit_transform(data)
    y_tot = data['y']

    errors = np.zeros(data.shape[0])

    # LOOCV
    for idx in range(data.shape[0]):
        mask = np.ones(len(data), dtype=bool)
        mask[idx] = False
        X_train = X_tot[mask]
        y_train = data['y'].values[mask]

        model = sm.OLS(y_train,X_train)
        results = model.fit()

        # prediction
        X_valid = X_tot[idx:idx+1]
        y_valid = data['y'].iloc[idx]
        pred_i = float(results.predict(X_valid).iloc[0])
        errors[idx] = (y_valid - pred_i)**2
    print('LOOCV for model degree: {0} = {1} \n'.format(deg,np.mean(errors)))



LOOCV for model degree: 1 = 8.96598993737657 

LOOCV for model degree: 2 = 0.9540287875907902 

LOOCV for model degree: 3 = 0.9563883817455859 

LOOCV for model degree: 4 = 0.9596109312225206 



* Repeating the experiment with a different seed produces a different dataset,
so the LOOCV errors change numerically. However, the ranking across models is
consistent: degree 2 still achieves the lowest error. The numerical differences
are expected — LOOCV has high variance as an estimator of the expected test
error, because the $n$ training sets overlap heavily (each shares $n-1$
observations), making the individual errors strongly correlated.


* The model with the lowest LOOCV error is **degree 2**, which is exactly what
we expected. The true DGP is:

$$Y = X - 2X^2 + \epsilon$$

Degrees 3 and 4 perform slightly worse because they estimate $\beta_3$ and
$\beta_4$, which are zero in the true model — this introduces extra variance
without any reduction in bias.